[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/africlimate-research/ACMAD_Seasonal_Calibration/blob/main/ACMAD_OND_IDR_App_Companion.ipynb)

# Companion notebook · Check the learning app yourself
**Goes with the interactive learning app (`app/`) and the original practical (`ACMAD_OND_IDR_Offline_Practical.ipynb`)**

The app shows the results; this notebook shows the code behind them. It recomputes the app's numbers and plots from the data files, using only NumPy, Matplotlib, scikit-learn and `idr_xarray`. **It does not import any of the app's code**, so it is an independent check.

### How to use it
- **Alongside the app.** Each section names the app page it checks, such as *App: Lab 4 · A*. Open that page and compare.
- **After the app.** Run it top to bottom (about a minute), then change the code and see what happens.
- **The checks.** Lines starting with ✓ give a number exactly as the app displays it at its default settings. A ✗ line shows both values. The last cell counts the matches.
- **Settings.** Section 1 uses the app's defaults, which are also the original notebook's: seed 42, 1,200 training pairs, 1,500 test pairs, train 2001–2015. If you change them here (or in the app's sidebar), the checks only print values; set the same values in both places to compare.

### What is in the original notebook and what is new
Sections 2–5 (Labs 1–4) follow the original notebook and reproduce its numbers. Section 6 (Lab 5) is **not in the original notebook**: the location-aware anomaly model, local cutoffs, leakage switches, shuffle controls and season resampling were added for the app. The original notebook asks the questions; this section runs the experiments.

## 0 — Setup (Colab only)
Run this first in Google Colab: it downloads the repository and installs the packages. On your own computer, skip it, and start Jupyter from the repository folder so that `data/` and `idr_xarray.py` are found.

In [ ]:
import os, sys
from pathlib import Path

if 'google.colab' in sys.modules:
    REPO = "https://github.com/africlimate-research/ACMAD_Seasonal_Calibration.git"
    project_path = Path("/content/acmad-idr-practical")
    if not project_path.exists():
        !git clone -q {REPO} {project_path}
    os.chdir(project_path)
    sys.path.insert(0, str(project_path))
    !pip install -q -r data/requirements.txt
print("Working folder:", Path.cwd())

## 1 — Data, settings and shared functions
Every later section uses the functions defined here. Read them once: they are the whole method in about 60 lines.

In [ ]:
import warnings
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.metrics import roc_auc_score, roc_curve

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    from idr_xarray import IDRModel

DATA = Path('data')
X = np.load(DATA / 'seas5_OND.npy')         # SEAS5 OND forecast (mm): 25 seasons x 12,424 locations
Y = np.load(DATA / 'chirps_obs_OND.npy')    # CHIRPS observed OND total (mm), same shape
YEARS = np.load(DATA / 'pred_years_OND.npy')
N_YEARS, N_LOC = X.shape
assert X.shape == Y.shape == (25, 12424) and np.array_equal(YEARS, np.arange(2001, 2026))
print(f'{N_YEARS} OND seasons ({YEARS[0]}–{YEARS[-1]}) x {N_LOC:,} locations')

**Settings.** These match the app's sidebar defaults and the original notebook.

In [ ]:
SEED = 42          # random seed for drawing the pairs
N_TRAIN = 1200     # training pairs (location-seasons) used to fit IDR
N_TEST = 1500      # test pairs used for scoring
SPLIT_YEAR = 2015  # train on seasons up to and including this year, test on the rest

DEFAULTS = (SEED, N_TRAIN, N_TEST, SPLIT_YEAR) == (42, 1200, 1500, 2015)
CATS = ['BN', 'NN', 'AN']
CHECKS = []


def check(where, label, shown, app):
    """Print a number formatted as the app shows it and compare it with the app (default settings only)."""
    if not DEFAULTS:
        print(f'  · {where} · {label}: {shown}')
        return
    ok = shown == app
    CHECKS.append((where, label, shown, app, ok))
    print(f"  {'✓' if ok else '✗'} {where} · {label}: {shown}" + ('' if ok else f'   (app shows {app})'))

**Shared functions.** IDR always predicts with "wetter forecast → wetter outcome" (`increasing=True`).

`cdf_at` reads each forecast's CDF at its own threshold. With one threshold for everyone it gives the same result as the original notebook's `pred.cdf_at(q1)` (checked in Section 4). The per-forecast version is needed later, when each location has its own cutoffs.

In [ ]:
def fit_idr(x, y):
    return IDRModel(increasing=True).fit(x, y)


def cdf_at(dist, t):
    """F_i(t_i): each forecast's predictive CDF at its own threshold (t: one number or one per forecast)."""
    n = dist.cdf.shape[0]
    t = np.broadcast_to(np.asarray(t, float), (n,))
    thr = np.concatenate([[-np.inf], dist.thresholds])
    cdf = np.hstack([np.zeros((n, 1)), dist.cdf])
    return cdf[np.arange(n), np.searchsorted(thr, t, side='right') - 1]


def tercile_probs(model, x, q1, q2):
    """(n, 3) array of P(BN), P(NN), P(AN) read off the predictive CDF at the two cutoffs."""
    d = model.predict(np.atleast_1d(x))
    a, b = cdf_at(d, q1), cdf_at(d, q2)
    return np.column_stack([a, b - a, 1 - b])


def categories(y, q1, q2):
    """0 = BN (y <= q1), 1 = NN, 2 = AN (y > q2). Same as np.digitize(y, [q1, q2], right=True)."""
    return (np.asarray(y) > q1).astype(int) + (np.asarray(y) > q2).astype(int)


def freqs(cat):
    return np.array([(cat == j).mean() for j in range(3)])


def brier(p, o):
    return float(np.mean((np.asarray(p, float) - np.asarray(o, float)) ** 2))


def bss(P, cat, ref):
    """Brier skill score per category. ref: 3 reference probabilities, or one row of 3 per forecast."""
    ref = np.broadcast_to(np.asarray(ref, float), P.shape)
    return np.array([1 - brier(P[:, j], cat == j) / brier(ref[:, j], cat == j) for j in range(3)])


def run_pooled(seed=SEED, n_train=N_TRAIN, n_test=N_TEST, split_year=SPLIT_YEAR):
    """The original notebook's experiment (cells 9, 13, 16), keeping each pair's location and year."""
    train_rows = YEARS <= split_year
    Xtr, Ytr, Xte, Yte = X[train_rows], Y[train_rows], X[~train_rows], Y[~train_rows]
    rng = np.random.default_rng(seed)
    it = rng.choice(Xtr.size, size=min(n_train, Xtr.size), replace=False)
    ie = rng.choice(Xte.size, size=min(n_test, Xte.size), replace=False)
    r = SimpleNamespace(
        train_rows=train_rows,
        x_train=Xtr.ravel()[it], y_train=Ytr.ravel()[it], loc_train=it % N_LOC,
        x_test=Xte.ravel()[ie], y_test=Yte.ravel()[ie], loc_test=ie % N_LOC,
        year_test=YEARS[~train_rows][ie // N_LOC])
    r.model = fit_idr(r.x_train, r.y_train)
    y_pool = Ytr.ravel()                                   # every training-year CHIRPS value
    r.q1, r.q2 = (float(v) for v in np.quantile(y_pool, [1 / 3, 2 / 3]))
    r.P = tercile_probs(r.model, r.x_test, r.q1, r.q2)
    r.cat = categories(r.y_test, r.q1, r.q2)
    r.clim = freqs(categories(y_pool, r.q1, r.q2))         # training-years climatology
    return r


r = run_pooled()
print(f'Fitted IDR on {len(r.x_train):,} pairs; {len(r.x_test):,} test pairs from {len(np.unique(r.year_test))} seasons')

## 2 — Meet the data (App: Lab 1)
### 2A · How closely does SEAS5 follow what fell?
An **anomaly** is how much wetter or drier a season was than that location's own 25-season average.

In [ ]:
r_raw = np.corrcoef(X.ravel(), Y.ravel())[0, 1]
Xa, Ya = X - X.mean(0), Y - Y.mean(0)          # anomalies
r_anom = np.corrcoef(Xa.ravel(), Ya.ravel())[0, 1]
print(f'Correlation over all {X.size:,} location-seasons')
check('Lab 1 · A', 'r, raw totals', f'{r_raw:.2f}', '0.94')
check('Lab 1 · A', 'r, anomalies', f'{r_anom:.2f}', '0.32')

pts = np.random.default_rng(2026).choice(N_LOC, size=600, replace=False)   # the 600 locations the app plots
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, (xx, yy, unit, rr) in zip(axes, [(X[:, pts], Y[:, pts], 'OND total (mm)', r_raw),
                                         (Xa[:, pts], Ya[:, pts], 'anomaly (mm)', r_anom)]):
    lim = np.quantile(np.abs(np.r_[xx.ravel(), yy.ravel()]), 0.995)
    lo = -lim if 'anomaly' in unit else 0
    ax.scatter(xx.ravel(), yy.ravel(), s=3, alpha=0.25, color='0.35')
    ax.plot([lo, lim], [lo, lim], 'k--', lw=1, label='Perfect forecast (1:1)')
    ax.set(xlim=(lo, lim), ylim=(lo, lim), aspect='equal', xlabel=f'SEAS5 {unit}', ylabel=f'CHIRPS {unit}',
           title=f'r = {rr:.2f} (all locations)')
    ax.legend(loc='upper left')
plt.tight_layout(); plt.show()

**Look for:** the raw totals hug the diagonal because wet places are wet in both datasets. Remove each location's average and most of the agreement goes.

### 2B · Is the forecast's error the same everywhere?
Sort every location-season by its SEAS5 value into ten equal groups (deciles), then average the bias SEAS5 − CHIRPS in each group.

In [ ]:
x, y = X.ravel(), Y.ravel()
d = x - y
edges = np.quantile(x, np.linspace(0, 1, 11))
k = np.clip(np.searchsorted(edges, x, side='right') - 1, 0, 9)
mean_bias = np.array([d[k == b].mean() for b in range(10)])

print(f"{'SEAS5 group (mm)':>18} {'mean bias (mm)':>15} {'SEAS5 avg':>10} {'CHIRPS avg':>11}")
for b in range(10):
    print(f'{edges[b]:8.0f}–{edges[b + 1]:<8.0f} {mean_bias[b]:+15.1f} {x[k == b].mean():10.0f} {y[k == b].mean():11.0f}')
check('Lab 1 · B', 'smallest group bias (mm)', f'{mean_bias.min():+.0f}', '-8')
check('Lab 1 · B', 'largest group bias (mm)', f'{mean_bias.max():+.0f}', '+91')
check('Lab 1 · B', 'overall mean bias (mm)', f'{d.mean():+.0f}', '+14')

fig, ax = plt.subplots(figsize=(9, 3.8))
labels = [f'{a:.0f}–{b:.0f}' for a, b in zip(edges[:-1], edges[1:])]
ax.plot(labels, mean_bias, 'o-', color='#eda100', lw=2, label='Average bias (SEAS5 − CHIRPS)')
ax.axhline(d.mean(), color='0.4', ls=':', label=f'One-number correction {d.mean():+.0f} mm')
ax.axhline(0, color='k', lw=0.8)
ax.set(xlabel='SEAS5 forecast group (mm)', ylabel='SEAS5 − CHIRPS (mm)')
ax.tick_params(axis='x', rotation=30); ax.legend(); plt.tight_layout(); plt.show()

### 2C · One location, 25 seasons
The year-to-year correlation r is computed across the 25 seasons at one location. The app's location menu picks the location whose average CHIRPS rainfall is closest to the 10th, 50th or 90th percentile across Africa.

In [ ]:
den = np.sqrt((Xa ** 2).sum(0) * (Ya ** 2).sum(0))
with np.errstate(invalid='ignore', divide='ignore'):
    r_loc = np.where(den > 0, (Xa * Ya).sum(0) / den, np.nan)   # NaN where a location never varies
mean_rain = Y.mean(0)


def location_by_wetness(pct):
    return int(np.argmin(np.abs(mean_rain - np.percentile(mean_rain, pct))))


PLACES = {'dry': location_by_wetness(10), 'typical': location_by_wetness(50), 'wet': location_by_wetness(90)}
EXPECT = {'dry': ('6995', '-0.08', '1 mm', '2 mm'),
          'typical': ('5204', '0.27', '54 mm', '52 mm'),
          'wet': ('1560', '0.48', '426 mm', '421 mm')}
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, (name, loc) in zip(axes, PLACES.items()):
    e = EXPECT[name]
    check('Lab 1 · C', f'{name} place: column', str(loc), e[0])
    check('Lab 1 · C', f'{name} place: year-to-year r', 'undefined' if np.isnan(r_loc[loc]) else f'{r_loc[loc]:.2f}', e[1])
    check('Lab 1 · C', f'{name} place: average CHIRPS', f'{Y[:, loc].mean():.0f} mm', e[2])
    check('Lab 1 · C', f'{name} place: average SEAS5', f'{X[:, loc].mean():.0f} mm', e[3])
    ax.axvspan(SPLIT_YEAR + 0.5, YEARS[-1] + 0.5, color='#2a78d6', alpha=0.07, label='Test years')
    ax.plot(YEARS, Y[:, loc], 'o-', color='k', ms=4, label='CHIRPS (what fell)')
    ax.plot(YEARS, X[:, loc], 'D-', color='#eda100', ms=4, label='SEAS5 (forecast)')
    ax.set(title=f'{name.capitalize()} place (column {loc}), r = {r_loc[loc]:.2f}', xlabel='Season', ylabel='mm')
axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

**Try it:** pick another column, e.g. `loc = 100`, and plot it the same way. How often does SEAS5 get the *direction* (wetter or drier than usual) right?

### 2D · Every location's year-to-year match
The same correlation for every location. This is the question a seasonal forecast must answer: *is this season wetter or drier than usual here?*

In [ ]:
valid = r_loc[~np.isnan(r_loc)]
check('Lab 1 · D', 'median r across locations', f'{np.median(valid):.2f}', '0.17')
check('Lab 1 · D', 'share of locations with r < 0', f'{(valid < 0).mean():.0%}', '25%')
check('Lab 1 · D', 'share of locations with r < 0.3', f'{(valid < 0.3).mean():.0%}', '71%')

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(valid, bins=np.arange(-1, 1.0001, 0.05), color='#2a78d6', edgecolor='white')
ax.axvline(np.median(valid), color='k', lw=2, label=f'Median r = {np.median(valid):.2f}')
ax.axvline(0, color='0.5', ls=':', label='No relationship')
ax.set(xlabel='Correlation across the 25 seasons at one location', ylabel='Number of locations')
ax.legend(); plt.tight_layout(); plt.show()

## 3 — How IDR works (App: Lab 2)
### 3A · One threshold at a time
For a threshold *t*, each training pair is 1 if CHIRPS stayed at or below *t* and 0 if it was wetter. IDR fits the share of 1s as a curve that **may only go down** as SEAS5 increases. The app starts *t* at the pooled lower tercile, rounded to whole millimetres.

In [ ]:
t = int(round(r.q1))
below = r.y_train <= t
dry_x, wet_x = np.quantile(r.x_train, [0.1, 0.9])            # a dry and a wet forecast
p_dry, p_wet = cdf_at(r.model.predict(np.array([dry_x, wet_x])), t)
check('Lab 2 · A', 'threshold t (mm)', str(t), '16')
check('Lab 2 · A', 'all training seasons <= t', f'{below.mean():.0%}', '36%')
check('Lab 2 · A', f'IDR, dry forecast ({dry_x:.0f} mm)', f'{p_dry:.0%}', '100%')
check('Lab 2 · A', f'IDR, wet forecast ({wet_x:.0f} mm)', f'{p_wet:.0%}', '0%')

xs = np.unique(r.x_train)
fitted = cdf_at(r.model.predict(xs), t)
print('The fitted curve never goes up:', bool(np.all(np.diff(fitted) <= 1e-12)))

jitter = np.random.default_rng(0).uniform(-0.05, 0.05, len(below))
fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(np.maximum(r.x_train, 1), below + jitter, s=8, alpha=0.35, color='0.35',
           label='Past seasons (1 = stayed <= t, 0 = wetter)')
ax.step(np.maximum(xs, 1), fitted, where='post', color='#2a78d6', lw=2.5, label='IDR: P(CHIRPS <= t | SEAS5)')
ax.set(xscale='log', xlabel='SEAS5 OND forecast (mm, log axis; values below 1 mm drawn at 1)',
       ylabel=f'Stayed at or below {t} mm?', ylim=(-0.1, 1.1))
ax.legend(); plt.tight_layout(); plt.show()

**About the log axis.** Equal distances on a log axis are equal *multiples* (1 → 10 → 100 → 1,000 mm), which spreads out the dry end where half the data lies. Change `xscale='log'` to `'linear'` to see the same curve squeezed against the left edge.

### 3B · PAVA by hand
IDR's curve comes from **pool adjacent violators (PAVA)**: walk from the driest forecast to the wettest, and whenever a wetter forecast shows a *higher* share of dry outcomes, replace the two with their weighted average. The app steps through this on 30 training seasons spread evenly from driest to wettest SEAS5. Here we write PAVA ourselves and compare it with `IDRModel` on the same 30 seasons.

In [ ]:
def pava_decreasing(values, weights):
    """Closest non-increasing sequence (weighted least squares). Returns the fit and the number of merges."""
    blocks, merges = [], 0                      # each block: [value, weight, number of groups]
    for v, w in zip(values, weights):
        blocks.append([v, w, 1])
        while len(blocks) > 1 and blocks[-2][0] < blocks[-1][0] - 1e-12:   # a wetter forecast is "drier": pool
            v2, w2, n2 = blocks.pop()
            v1, w1, n1 = blocks.pop()
            blocks.append([(v1 * w1 + v2 * w2) / (w1 + w2), w1 + w2, n1 + n2])
            merges += 1
    return np.concatenate([[v] * n for v, w, n in blocks]), merges


order = np.argsort(r.x_train)
pick = order[np.linspace(0, len(order) - 1, 30).astype(int)]
px, py = r.x_train[pick], r.y_train[pick]
ux, inv, cnt = np.unique(px, return_inverse=True, return_counts=True)
raw = np.bincount(inv, weights=(py <= t).astype(float)) / cnt      # share <= t at each distinct SEAS5 value
by_hand, merges = pava_decreasing(raw, cnt)
by_idr = cdf_at(fit_idr(px, py).predict(ux), t)
print('Raw shares :', np.round(raw, 2))
print('PAVA fit   :', np.round(by_hand, 2))
print('By-hand PAVA equals IDRModel:', np.allclose(by_hand, by_idr, atol=1e-3))
check('Lab 2 · B', 'pooling steps needed', str(merges), '2')

**Try it:** set `t = 100` and rerun the cell. More violations appear, and the hand-written PAVA still matches `IDRModel`.

### 3C · The full answer for one forecast: the predictive CDF
Fix one SEAS5 value (the app starts at the median training forecast) and read IDR's whole distribution. The comparison is a straight-line regression with a bell-shaped (Gaussian) error.

In [ ]:
x_val = int(np.median(r.x_train))
d1 = r.model.predict(np.array([x_val]))
q10, q50, q90 = np.atleast_1d(d1.quantile(np.array([0.1, 0.5, 0.9])))

b_lin, a_lin = np.polyfit(r.x_train, r.y_train, 1)                       # y = a + b x + error
sigma = np.std(r.y_train - (a_lin + b_lin * r.x_train), ddof=2)
lg_cdf = lambda tt: norm.cdf((np.asarray(tt) - a_lin - b_lin * x_val) / sigma)

check('Lab 2 · C', 'SEAS5 forecast (mm)', str(x_val), '43')
check('Lab 2 · C', 'IDR median', f'{q50:.0f} mm', '33 mm')
check('Lab 2 · C', 'IDR 10–90% range', f'{q10:.0f}–{q90:.0f} mm', '14–80 mm')
check('Lab 2 · C', 'Linear+Gaussian median', f'{a_lin + b_lin * x_val:.0f} mm', '44 mm')
check('Lab 2 · C', 'Linear+Gaussian P(rain < 0)', f'{float(lg_cdf(0.0)):.1%}', '21.5%')

tt = np.linspace(min(0, a_lin + b_lin * x_val - 2.6 * sigma), 300, 500)
fig, ax = plt.subplots(figsize=(9, 4))
ax.step(np.r_[tt[0], d1.thresholds], np.r_[0, d1.cdf[0]], where='post', color='#2a78d6', lw=2.5, label='IDR')
ax.plot(tt, lg_cdf(tt), '--', color='#e87ba4', lw=2.5, label='Linear + Gaussian')
ax.axvspan(tt[0], 0, color='#e87ba4', alpha=0.08, label='Negative rain (impossible)')
ax.set(xlim=(tt[0], 300), ylim=(0, 1.02), xlabel='Rainfall threshold (mm)', ylabel='P(CHIRPS <= threshold)',
       title=f'Predictive CDF for a SEAS5 forecast of {x_val} mm')
ax.legend(); plt.tight_layout(); plt.show()

### 3D · The original notebook's four example forecasts
The curves for a dry, two middling and a wet forecast (10th, 40th, 70th and 90th percentiles of the training forecasts). Wetter forecasts must never cross above drier ones.

In [ ]:
example_x = np.quantile(r.x_train, [.1, .4, .7, .9])
dist = r.model.predict(example_x)
fig, ax = plt.subplots(figsize=(8, 4))
for i, xx in enumerate(example_x):
    ax.step(dist.thresholds, dist.cdf[i], where='post', label=f'SEAS5: {xx:.0f} mm')
ax.axvline(t, color='0.5', ls=':')
ax.set(xlabel='Observed rainfall threshold (mm)', ylabel='P(CHIRPS <= threshold)', ylim=(0, 1.02), xlim=(0, 600))
ax.legend(); plt.tight_layout(); plt.show()

p_at_t = cdf_at(dist, t)
for xx, p in zip(example_x, p_at_t):
    print(f'SEAS5 {xx:5.0f} mm: P(CHIRPS <= {t} mm) = {p:.2f}')
print('Curves never cross (each wetter CDF lies on or below the drier one):',
      bool(np.all(np.diff(dist.cdf, axis=0) <= 1e-12)))

## 4 — From curve to terciles (App: Lab 3)
The pooled cutoffs split all training-year CHIRPS values into three equal groups. Then P(BN) = F(q⅓), P(NN) = F(q⅔) − F(q⅓) and P(AN) = 1 − F(q⅔).

In [ ]:
check('Lab 3', 'pooled lower cutoff q1/3 (mm)', f'{r.q1:.1f}', '15.8')
check('Lab 3', 'pooled upper cutoff q2/3 (mm)', f'{r.q2:.1f}', '146.9')

dist_test = r.model.predict(r.x_test)
print('cdf_at matches the original notebook\'s pred.cdf_at:',
      bool(np.allclose(cdf_at(dist_test, r.q1), np.atleast_1d(dist_test.cdf_at(r.q1)))))
assert np.all(r.P >= -1e-8) and np.allclose(r.P.sum(1), 1)

print('\nAcross all held-out forecasts (App: Lab 3, last table)')
for j, c in enumerate(CATS):
    check('Lab 3', f'{c}: average forecast probability', f'{r.P[:, j].mean():.3f}', {'BN': '0.339', 'NN': '0.319', 'AN': '0.342'}[c])
    check('Lab 3', f'{c}: observed frequency', f'{(r.cat == j).mean():.3f}', {'BN': '0.319', 'NN': '0.345', 'AN': '0.336'}[c])

**One forecast at the typical place.** The app's Lab 3 opens at the typical place, with the forecast set to that location's average training-year SEAS5.

In [ ]:
loc = PLACES['typical']
x_loc = int(round(X[r.train_rows, loc].mean()))
P1 = tercile_probs(r.model, x_loc, r.q1, r.q2)[0]
check('Lab 3', 'typical place: SEAS5 forecast (mm)', str(x_loc), '48')
check('Lab 3', 'typical place: tercile probabilities', ' · '.join(f'{c} {p:.0%}' for c, p in zip(CATS, P1)), 'BN 14% · NN 84% · AN 2%')

**Why *which* cutoffs matter.** At the dry place, compare the pooled cutoffs with the location's own training-year terciles. Choose *Dry place* and switch the cutoffs in the app to see the same numbers.

In [ ]:
Ytr = Y[r.train_rows]
LQ1, LQ2 = np.quantile(Ytr, [1 / 3, 2 / 3], axis=0)          # each location's own training-year terciles

loc = PLACES['dry']
counts = lambda c: ' · '.join(f'{k} {int((c == j).sum())}' for j, k in enumerate(CATS))
check('Lab 3', 'dry place, training seasons by pooled cutoffs', counts(categories(Ytr[:, loc], r.q1, r.q2)),
      'BN 15 · NN 0 · AN 0')
check('Lab 3', 'dry place, training seasons by local cutoffs', counts(categories(Ytr[:, loc], LQ1[loc], LQ2[loc])),
      'BN 5 · NN 5 · AN 5')
x_te, y_te = X[~r.train_rows, loc], Y[~r.train_rows, loc]
for mode, (a, b) in [('pooled', (r.q1, r.q2)), ('local', (LQ1[loc], LQ2[loc]))]:
    Pl = tercile_probs(r.model, x_te, a, b)
    hits = int((categories(y_te, a, b) == Pl.argmax(1)).sum())
    check('Lab 3', f'dry place, most likely category right ({mode} cutoffs)', f'{hits} of {len(y_te)}',
          {'pooled': '10 of 10', 'local': '5 of 10'}[mode])

With pooled cutoffs every season at the dry place is "below normal", so always saying BN is always right. That shows the place is dry, not that the forecast knows this season.

## 5 — Is the forecast any good? (App: Lab 4)
### 5A · Brier score and Brier skill score
BS = mean (p − o)², lower is better. BSS = 1 − BS(IDR) / BS(climatology); above 0 beats climatology.

In [ ]:
print(f"{'Category':<9} {'IDR BS':>8} {'Clim BS':>8} {'BSS':>8}")
B = bss(r.P, r.cat, r.clim)
EXP_A = {'BN': ('0.060', '0.217', '+0.723'), 'NN': ('0.109', '0.226', '+0.517'),
         'AN': ('0.050', '0.223', '+0.774')}
for j, c in enumerate(CATS):
    o = (r.cat == j).astype(float)
    bs_i, bs_c = brier(r.P[:, j], o), brier(np.full(len(o), r.clim[j]), o)
    print(f'{c:<9} {bs_i:8.4f} {bs_c:8.4f} {B[j]:+8.3f}')
for j, c in enumerate(CATS):
    o = (r.cat == j).astype(float)
    check('Lab 4 · A', f'{c}: Brier score, IDR', f'{brier(r.P[:, j], o):.3f}', EXP_A[c][0])
    check('Lab 4 · A', f'{c}: Brier score, climatology', f'{brier(np.full(len(o), r.clim[j]), o):.3f}', EXP_A[c][1])
    check('Lab 4 · A', f'{c}: BSS', f'{B[j]:+.3f}', EXP_A[c][2])

**Murphy decomposition.** Group the forecasts into 8 equal-width probability bins. The binned Brier score splits exactly into reliability − resolution + uncertainty. The "within-bin" term is the small difference from the unbinned score. The app's default category is AN.

In [ ]:
def bin_index(p, n_bins=8):
    return np.clip(np.floor(np.asarray(p) * n_bins).astype(int), 0, n_bins - 1)


def murphy(p, o, n_bins=8):
    k = bin_index(p, n_bins)
    obar, n = o.mean(), len(p)
    rel = res = 0.0
    p_binned = np.empty_like(p)
    for b in np.unique(k):
        m = k == b
        p_binned[m] = p[m].mean()
        rel += m.sum() * (p[m].mean() - o[m].mean()) ** 2
        res += m.sum() * (o[m].mean() - obar) ** 2
    return dict(reliability=rel / n, resolution=res / n, uncertainty=obar * (1 - obar),
                bs_raw=brier(p, o), bs_binned=brier(p_binned, o))


o_an = (r.cat == 2).astype(float)
M = murphy(r.P[:, 2], o_an)
for key, app in [('reliability', '0.0003'), ('resolution', '0.1728'),
                 ('uncertainty', '0.2231'), ('bs_raw', '0.0505'),
                 ('bs_binned', '0.0506')]:
    check('Lab 4 · A', f'AN Murphy: {key}', f'{M[key]:.4f}', app)
check('Lab 4 · A', 'AN Murphy: within-bin term', f"{M['bs_raw'] - M['bs_binned']:.4f}", '-0.0002')
print('Binned BS = REL − RES + UNC:',
      bool(np.isclose(M['bs_binned'], M['reliability'] - M['resolution'] + M['uncertainty'])))

### 5B · Reliability: when IDR says 70%, does it happen 70% of the time?
The original notebook dropped bins with fewer than 10 forecasts. The app keeps them and draws them hollow.

In [ ]:
def reliability(p, o, n_bins=8):
    k = bin_index(p, n_bins)
    keep = [b for b in range(n_bins) if (k == b).any()]
    return (np.array([p[k == b].mean() for b in keep]), np.array([o[k == b].mean() for b in keep]),
            np.array([(k == b).sum() for b in keep]))


mean_p, freq, count = reliability(r.P[:, 2], o_an)
check('Lab 4 · B', 'AN forecasts per bin', str([int(c) for c in count]), '[878, 38, 16, 44, 100, 84, 340]')
for j, c in enumerate(CATS):
    cnt_j = reliability(r.P[:, j], (r.cat == j).astype(float))[2]
    check('Lab 4 · B', f'{c}: bins with fewer than 10 forecasts', f'{int((cnt_j < 10).sum())} of {len(cnt_j)}',
          {'BN': '1 of 8', 'NN': '0 of 8', 'AN': '0 of 7'}[c])

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot([0, 1], [0, 1], 'k--', label='Perfect reliability')
ax.plot(mean_p, freq, '-', color='#2a78d6')
ax.scatter(mean_p[count >= 10], freq[count >= 10], color='#2a78d6', zorder=3, label='IDR (AN)')
ax.scatter(mean_p[count < 10], freq[count < 10], facecolor='white', edgecolor='#2a78d6', zorder=3,
           label='Fewer than 10 forecasts')
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel='Forecast probability of above normal', ylabel='Observed frequency')
ax.legend(); plt.tight_layout(); plt.show()

### 5C · Discrimination (ROC)
AUC measures how well the forecast ranks event seasons above the rest. It ignores whether the probabilities are calibrated.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
for j, c in enumerate(CATS):
    o = (r.cat == j).astype(int)
    auc = roc_auc_score(o, r.P[:, j])
    fpr, tpr, _ = roc_curve(o, r.P[:, j])
    ax.plot(fpr, tpr, lw=2, label=f'{c} (AUC = {auc:.3f})')
    check('Lab 4 · C', f'{c}: ROC AUC', f'{auc:.3f}', {'BN': '0.973', 'NN': '0.916', 'AN': '0.979'}[c])
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Chance')
ax.set(xlabel='False alarm rate', ylabel='Hit rate'); ax.legend(); plt.tight_layout(); plt.show()

warn = r.P[:, 2] >= 0.5          # the app's default: warn of AN when P(AN) >= 0.5
o = o_an.astype(bool)
cm = (int((warn & o).sum()), int((~warn & o).sum()), int((warn & ~o).sum()), int((~warn & ~o).sum()))
check('Lab 4 · C', 'AN at cutoff 0.5: hits / misses / false alarms / correct quiet', ' / '.join(map(str, cm)),
      '463 / 41 / 61 / 935')

### 5D · How much would these numbers move with different luck?
Two kinds of rerun, with the middle 90% of results:
- **different random pairs:** the same experiment with 50 other seeds (43–92);
- **a different mix of seasons:** keep the model and resample the 10 test seasons with replacement, 500 times.

In [ ]:
seed_bss = np.array([bss(rr.P, rr.cat, rr.clim) for rr in (run_pooled(seed=SEED + 1 + k) for k in range(50))])


def year_bootstrap(P, cat, ref, years, n_boot=500, seed=0):
    rng = np.random.default_rng(seed)
    ref = np.broadcast_to(np.asarray(ref, float), P.shape)
    uy = np.unique(years)
    idx = {yy: np.flatnonzero(years == yy) for yy in uy}
    out = []
    for _ in range(n_boot):
        i = np.concatenate([idx[yy] for yy in rng.choice(uy, size=len(uy), replace=True)])
        out.append(bss(P[i], cat[i], ref[i]))
    return np.array(out)


def bss_by_year(P, cat, ref, years):
    ref = np.broadcast_to(np.asarray(ref, float), P.shape)
    uy = np.unique(years)
    return uy, np.array([bss(P[years == yy], cat[years == yy], ref[years == yy]) for yy in uy])


yb = year_bootstrap(r.P, r.cat, r.clim, r.year_test)
check('Lab 4 · D', 'AN, different random pairs (5–95%)', '{:.2f} to {:.2f}'.format(*np.percentile(seed_bss[:, 2], [5, 95])),
      '0.72 to 0.78')
check('Lab 4 · D', 'AN, different mix of seasons (5–95%)', '{:.2f} to {:.2f}'.format(*np.percentile(yb[:, 2], [5, 95])),
      '0.74 to 0.80')
uy, per_year = bss_by_year(r.P, r.cat, r.clim, r.year_test)
check('Lab 4 · D', 'AN, worst season', f'{uy[np.argmin(per_year[:, 2])]} ({per_year[:, 2].min():+.2f})', '2022 (+0.67)')
check('Lab 4 · D', 'AN, best season', f'{uy[np.argmax(per_year[:, 2])]} ({per_year[:, 2].max():+.2f})', '2023 (+0.87)')

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.bar(uy, per_year[:, 2], color='#1baf7a')
ax.axhline(B[2], color='k', ls='--', lw=1, label='All seasons together')
ax.set(xlabel='Test season', ylabel='BSS for AN, that season only', xticks=uy); ax.legend()
plt.tight_layout(); plt.show()

## 6 — Traps and illusions (App: Lab 5) · *not in the original notebook*
The original notebook raises these questions in its discussion and take-home sections. This section runs the experiments the app uses to answer them.

### 6A · The pooling illusion (App: Lab 5, tab A)
Three ways to score the **same** test pairs:
1. **① The original notebook:** pooled model, pooled cutoffs, pooled climatology.
2. **② Same model, local cutoffs:** "below normal" now means drier than usual *for this location*; the reference is that location's own training-year frequencies.
3. **③ Location-aware (anomaly) model:** standardise each location's SEAS5 and CHIRPS with its own training-year mean and standard deviation, fit IDR on those anomalies, and score against local cutoffs.

Pairs are left out of all three rows when the location's two cutoffs are equal (many identical seasons, e.g. always 0 mm) or its training CHIRPS spread is 1 mm or less.

In [ ]:
Xtr = X[r.train_rows]
mx, sx, my, sy = Xtr.mean(0), Xtr.std(0), Ytr.mean(0), Ytr.std(0)
lcat = categories(Ytr, LQ1, LQ2)
LREF = np.stack([(lcat == j).mean(0) for j in range(3)], axis=1)     # local training-year frequencies (L, 3)
good = (sy > 1) & (sx > 0) & (LQ1 < LQ2)                              # locations where ③ can be fitted

gt = good[r.loc_train]
lt_ = r.loc_train[gt]
model_a = fit_idr((r.x_train[gt] - mx[lt_]) / sx[lt_], (r.y_train[gt] - my[lt_]) / sy[lt_])

ok = good[r.loc_test]
le = r.loc_test[ok]
a1 = bss(r.P[ok], r.cat[ok], r.clim)
a2 = bss(tercile_probs(r.model, r.x_test[ok], LQ1[le], LQ2[le]), categories(r.y_test[ok], LQ1[le], LQ2[le]), LREF[le])
P3 = tercile_probs(model_a, (r.x_test[ok] - mx[le]) / sx[le], (LQ1[le] - my[le]) / sy[le], (LQ2[le] - my[le]) / sy[le])
a3 = bss(P3, categories(r.y_test[ok], LQ1[le], LQ2[le]), LREF[le])

check('Lab 5 · A', 'test pairs scored in every row', f'{ok.sum():,}', '1,296')
check('Lab 5 · A', 'test pairs left out', str(int((~ok).sum())), '204')
check('Lab 5 · A', 'training pairs for the anomaly model', f'{gt.sum():,}', '1,020')
EXP_5A = {'BN': ('+0.63', '-0.19', '+0.03'), 'NN': ('+0.49', '-0.03', '+0.02'),
          'AN': ('+0.76', '-0.18', '+0.05')}
print(f"\n{'':<46}{'BN':>7}{'NN':>7}{'AN':>7}")
for name, v in [('① pooled model, pooled cutoffs', a1), ('② same model, local cutoffs', a2),
                ('③ anomaly model, local cutoffs', a3)]:
    print(f'{name:<46}' + ''.join(f'{x:+7.2f}' for x in v))
for j, c in enumerate(CATS):
    for i, v in enumerate([a1, a2, a3]):
        check('Lab 5 · A', f'{c}: setting {"①②③"[i]}', f'{v[j]:+.2f}', EXP_5A[c][i])

**Why ① on these pairs (+0.63 for BN) is lower than the original notebook's +0.72.** The left-out pairs are almost rainless places, which the pooled model scores very easily:

In [ ]:
a1_all, a1_out = bss(r.P, r.cat, r.clim), bss(r.P[~ok], r.cat[~ok], r.clim)
for j, c in enumerate(CATS):
    check('Lab 5 · A', f'{c}: ① on all {len(ok):,} pairs (the notebook)', f'{a1_all[j]:+.2f}',
          {'BN': '+0.72', 'NN': '+0.52', 'AN': '+0.77'}[c])
    check('Lab 5 · A', f'{c}: ① on the left-out pairs only', f'{a1_out[j]:+.2f}',
          {'BN': '+0.97', 'NN': '+0.87', 'AN': '+1.00'}[c])
check('Lab 5 · A', 'median average rainfall at left-out pairs', f'{np.median(mean_rain[r.loc_test[~ok]]):.1f} mm', '0.6 mm')

### 6B · Leakage (App: Lab 5, tab B)
Score the same test pairs while letting test-year information leak into one part at a time:
- **cutoffs** from all 25 years (changes *which event* is scored);
- **climatology** from all 25 years (changes *the yardstick*);
- **the model** trained on the test pairs too (it sees the answers).

"Local" uses the anomaly model ③. Its locations must have distinct cutoffs under both the training-year and the all-year definitions, so every variant scores the same pairs.

In [ ]:
def leakage(local, leak_cutoffs=False, leak_reference=False, leak_model=False):
    every = np.ones(N_YEARS, bool)
    cut_rows = every if leak_cutoffs else r.train_rows
    ref_rows = every if leak_reference else r.train_rows
    xf, yf, lf = r.x_train, r.y_train, r.loc_train
    if leak_model:
        xf, yf, lf = np.r_[xf, r.x_test], np.r_[yf, r.y_test], np.r_[lf, r.loc_test]
    if not local:
        q1, q2 = np.quantile(Y[cut_rows].ravel(), [1 / 3, 2 / 3])
        model = fit_idr(xf, yf) if leak_model else r.model
        P = tercile_probs(model, r.x_test, q1, q2)
        return bss(P, categories(r.y_test, q1, q2), freqs(categories(Y[ref_rows].ravel(), q1, q2)))
    A1, A2 = np.quantile(Y, [1 / 3, 2 / 3], axis=0)                        # all-year local cutoffs
    ok_l = (sy > 1) & (sx > 0) & (LQ1 < LQ2) & (A1 < A2)
    Q1, Q2 = np.quantile(Y[cut_rows], [1 / 3, 2 / 3], axis=0)
    rc = categories(Y[ref_rows], Q1, Q2)
    REF = np.stack([(rc == j).mean(0) for j in range(3)], axis=1)
    g = ok_l[lf]
    model = fit_idr((xf[g] - mx[lf[g]]) / sx[lf[g]], (yf[g] - my[lf[g]]) / sy[lf[g]])  # training-year statistics
    m = ok_l[r.loc_test]
    l = r.loc_test[m]
    P = tercile_probs(model, (r.x_test[m] - mx[l]) / sx[l], (Q1[l] - my[l]) / sy[l], (Q2[l] - my[l]) / sy[l])
    return SimpleNamespace(bss=bss(P, categories(r.y_test[m], Q1[l], Q2[l]), REF[l]),
                           P=P, cat=categories(r.y_test[m], Q1[l], Q2[l]), ref=REF[l], year=r.year_test[m])


EXP_5B = {('pooled', 'none'): '+0.723 · +0.517 · +0.774', ('pooled', 'leak_cutoffs'): '+0.731 · +0.512 · +0.771',
          ('pooled', 'leak_reference'): '+0.723 · +0.516 · +0.774', ('pooled', 'leak_model'): '+0.740 · +0.538 · +0.781',
          ('local', 'none'): '+0.027 · +0.023 · +0.047', ('local', 'leak_cutoffs'): '+0.030 · +0.038 · +0.054',
          ('local', 'leak_reference'): '-0.152 · -0.085 · -0.079', ('local', 'leak_model'): '+0.067 · +0.072 · +0.062'}
for setting in ['pooled', 'local']:
    for leak in ['none', 'leak_cutoffs', 'leak_reference', 'leak_model']:
        out = leakage(setting == 'local', **({} if leak == 'none' else {leak: True}))
        v = out.bss if setting == 'local' else out
        check('Lab 5 · B', f'{setting}, {leak.replace("_", " ")}: BSS BN · NN · AN', ' · '.join(f'{x:+.3f}' for x in v),
              EXP_5B[(setting, leak)])

**Look for:** in the pooled setting every leak is tiny, because 12,424 locations average out. In the local setting a leaked climatology makes the honest model look much **worse** (BN +0.03 → −0.15), and a leaked model makes it look **better**.

### 6C · Can calibration create skill? (App: Lab 5, tab C)
Shuffle the training SEAS5 values so they no longer match their seasons, refit IDR, and score on the same real test pairs, 100 times. Three controls:
- force the order the **wrong** way, by fitting on −SEAS5;
- shuffle the **test** SEAS5 too (reusing each shuffled model);
- **no SEAS5 at all:** always forecast the training climatology.

This refits IDR 200 times and takes a few seconds.

In [ ]:
right, wrong, both = [], [], []
for k in range(100):
    xs_ = r.x_train.copy()
    np.random.default_rng(k).shuffle(xs_)
    m = fit_idr(xs_, r.y_train)
    right.append(bss(tercile_probs(m, r.x_test, r.q1, r.q2), r.cat, r.clim))
    wrong.append(bss(tercile_probs(fit_idr(-xs_, r.y_train), -r.x_test, r.q1, r.q2), r.cat, r.clim))
    xe = r.x_test.copy()
    np.random.default_rng(1000 + k).shuffle(xe)
    both.append(bss(tercile_probs(m, xe, r.q1, r.q2), r.cat, r.clim))
fit_clim = freqs(categories(r.y_train, r.q1, r.q2))
const = bss(np.tile(fit_clim, (len(r.y_test), 1)), r.cat, r.clim)

check('Lab 5 · C', 'BN, real SEAS5', f'{B[0]:+.3f}', '+0.723')
check('Lab 5 · C', 'BN, shuffled SEAS5 (mean of 100)', f'{np.mean(right, 0)[0]:+.3f}', '+0.042')
check('Lab 5 · C', 'BN, wrong direction (mean of 100)', f'{np.mean(wrong, 0)[0]:+.3f}', '-0.073')
check('Lab 5 · C', 'BN, shuffled train and test (mean of 100)', f'{np.mean(both, 0)[0]:+.3f}',
      '-0.011')
check('Lab 5 · C', 'BN, always climatology', f'{const[0]:+.3f}', '-0.008')

fig, ax = plt.subplots(figsize=(8, 3.5))
for vals, lab in [(right, 'Shuffled SEAS5'), (wrong, 'Wrong direction'), (both, 'Shuffled train and test')]:
    ax.hist(np.array(vals)[:, 0], bins=20, alpha=0.6, label=lab)
ax.axvline(const[0], color='k', ls=':', label='Always climatology')
ax.set(xlabel='BSS for BN', ylabel='Number of shuffles'); ax.legend(); plt.tight_layout(); plt.show()

**Look for:** the shuffled score is small but not zero. IDR still ranks the test forecasts by their SEAS5 value, which says a lot about the *place*. Forcing the wrong order makes the score negative, and shuffling the test values too removes the gain.

### 6D · Reliable is not the same as skilful (App: Lab 5, tab D)
Compare IDR with a forecaster who **always** issues the training climatology, for BN.

In [ ]:
o_bn = (r.cat == 0).astype(float)
for name, f, app in [('IDR', r.P[:, 0], 'REL 0.002 · RES 0.158 · BS 0.060 · BSS +0.723'), ('always climatology', np.full(len(o_bn), r.clim[0]), 'REL 0.000 · RES 0.000 · BS 0.217 · BSS +0.000')]:
    M = murphy(f, o_bn)
    shown = (f"REL {M['reliability']:.3f} · RES {M['resolution']:.3f} · BS {M['bs_raw']:.3f} · "
             f"BSS {1 - M['bs_raw'] / brier(np.full(len(o_bn), r.clim[0]), o_bn):+.3f}")
    check('Lab 5 · D', name, shown, app)
check('Lab 5 · D', 'climatology always says (BN)', f'{r.clim[0]:.1%}', '33.3%')
check('Lab 5 · D', 'BN happened in the test seasons', f'{o_bn.mean():.1%}', '31.9%')

Climatology's reliability term is almost zero: its one probability is close to how often BN happened. Its resolution is exactly zero, because it never changes. It is reliable but has no skill.

### 6E · Not 1,500 independent cases (App: Lab 5, tab E)
The test pairs come from only 10 seasons, and locations in the same season share that season's weather. Compare two ways of putting an uncertainty range on the BN score:
- **resample pairs** (500 times), as if every pair were independent;
- **resample whole seasons** (500 times), keeping each season's pairs together.

In [ ]:
def pair_bootstrap(P, cat, ref, n_boot=500, seed=0):
    rng = np.random.default_rng(seed)
    ref = np.broadcast_to(np.asarray(ref, float), P.shape)
    out = []
    for _ in range(n_boot):
        i = rng.integers(0, len(cat), len(cat))
        out.append(bss(P[i], cat[i], ref[i]))
    return np.array(out)


Lc = leakage(True)
EXP_5E = {'pooled': ('+0.686 to +0.757', '+0.672 to +0.768', '+0.50 to +0.83, 0 below 0'),
          'local': ('+0.001 to +0.055', '-0.006 to +0.056', '-0.09 to +0.17, 4 below 0')}
for name, (P, cat, ref, yrs) in [('pooled', (r.P, r.cat, np.broadcast_to(r.clim, r.P.shape), r.year_test)),
                                 ('local', (Lc.P, Lc.cat, Lc.ref, Lc.year))]:
    pb, ybs = pair_bootstrap(P, cat, ref)[:, 0], year_bootstrap(P, cat, ref, yrs)[:, 0]
    uy, py = bss_by_year(P, cat, ref, yrs)
    check('Lab 5 · E', f'{name}, BN, resample pairs (5–95%)', '{:+.3f} to {:+.3f}'.format(*np.percentile(pb, [5, 95])),
          EXP_5E[name][0])
    check('Lab 5 · E', f'{name}, BN, resample seasons (5–95%)', '{:+.3f} to {:+.3f}'.format(*np.percentile(ybs, [5, 95])),
          EXP_5E[name][1])
    check('Lab 5 · E', f'{name}, BN, one season at a time',
          f'{py[:, 0].min():+.2f} to {py[:, 0].max():+.2f}, {int((py[:, 0] < 0).sum())} below 0', EXP_5E[name][2])
    if name == 'local':
        fig, ax = plt.subplots(figsize=(8, 3.2))
        ax.bar(uy, py[:, 0], color='#eb6834')
        ax.axhline(0, color='k', lw=0.8)
        ax.set(xlabel='Test season', ylabel='Local BSS for BN', xticks=uy)
        plt.tight_layout(); plt.show()

**Look for:** with the anomaly model, the season-by-season scores swing from about −0.09 to +0.17 and several seasons are below zero. A single overall number hides that.

## 7 — Summary of the checks

In [ ]:
if DEFAULTS:
    n_ok = sum(c[-1] for c in CHECKS)
    print(f'{n_ok} of {len(CHECKS)} numbers match the app exactly as it displays them.')
    for where, label, shown, app, ok in CHECKS:
        if not ok:
            print(f'  ✗ {where} · {label}: notebook {shown}, app {app}')
else:
    print('Custom settings: no expected values. Set the same values in the app sidebar and compare by eye.')

### Things to try
1. Set `SPLIT_YEAR = 2020` in Section 1 and rerun everything, and set the same split in the app's sidebar. Which conclusions survive?
2. Change `SEED`. How much does the BN BSS move compared with the season-to-season spread in 6E?
3. In 6A, fit the anomaly model with the supplied WMO means and standard deviations (`data/y_mean_wmo_OND.npy`, `data/y_std_wmo_OND.npy`) for CHIRPS instead of the training-year ones. Does the local skill change?
4. In 6C, replace the shuffle with SEAS5 values from a different location. Is any skill left?